# General

**Name:**  Clifton-John Walle and Meynaghi Aghdam, A.

**Title of the experiment:** Free-fall 

**Starting date:**  September 11th 2026 09:00

**Expected enddate:**  September 14th 2026 20:30

**TA:**  Christian 

**Goal of the experiment:**  Determination of the gravitational acceleration by dropping a ball and measuring the time 

**Research question:**  What is the gravitational acceleration, from a classical point of view, in the Applied Sciences Building of TU Delft within 4 significant figures?

**Expectations or Hypothesis:**  Expectation is to find a value close to $9.8125 m/s^2$

**Desired accuracy:**  up to the 4th significant figure and within 0.1% of the known literature value

**Repository with the used data and code**: https://github.com/cwalle123/Modern_Physics_IEP_Experiment_1.git



In [2]:
#import necessary libraries
import csv
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import norm
from matplotlib.ticker import MaxNLocator

# Preparation
 


**Assignments:**  

**Method:**
We chose the free-fall experiment to determine the gravitational acceleration. We did trial runs with the stopwatch on the "clock" app on the iphone, then started using the intended setup of the experiment for the same measurements. 

**Theory:**  
The main theoretical components are Newton's gravitational model and the laws of motion, given by:

$y = y0 + v\cdot t + \frac{g \cdot t^2}{2}$

One of the assumptions we make is that the drag does not affect the motion of the steel ball we experiment with. The other, less consequential, assumption is that the gravitational acceleration is constant across the distance traveled (0.5 - 2m). 

**Independent variable:**  
Time, t 

**Dependent variable:**  
Position of the ball, y 
Velocity of the ball, v 

**Controlled variable:**  
Gravitational acceleration, g 

**Measurement instruments \& Settings:**  
Two time-measurement sensors are used that detect the ball when it passes by. Such that we shall work with 2 position-time coordinates. We also have information regarding the position of the ball when it has no velocity (v = 0). The heights $y_1$, $y_2$, and $y_3$ were measured using a measuring tape with millimeter accuracy. The photoelectric sensors measure with microsecond accuracy; however, the time measurements were done with an Arduino Uno and running code. Based on this, the uncertainty of the time measurement was estimated to be a conservative 2 microseconds. 

**Procedure:**  
1. Turn on the magnet 
2. Attach the steel ball to the magnet
3. Make sure the first sensor was not already activated 
4. Turn off the magnet 
5. Be alert regarding the moment the sensors are activated while the ball falls 
6. Read the time registered between the 2 sensors in the serial monitor and collect it in the data file 

Do this procedure for different heights of the ball 


**Setup drawing (Figure 1 of the Report):**


<img src="experimental_setup.png" />


**Notes:**  
Some issues we encountered during our trials: 
1. The frame to which the sensors were attached was slightly tilted. Fortunately, there was an easy way to recalibrate the frame angle. 
2. The sensor at the bottom was supposed to be made of 2 sensors such that they covered a bigger part of the possible trajectories of the ball. Unfortunately only one of the two sensors worked for the bottom half. This problem made us tilt the sensor such that the ball was detected, leading to slight adjustments in the distance from the ground to the sensor, as well as multiple failed trials due to the ball not being detected at the bottom.



# Execution

In [ ]:
# Measurements: Can be found in the Data/data.csv file present in the repository.

**Observations:** The three different lengths were the ones to give valuable data, while other trials were determined to be unusable due to high errors in distance measurement


# Processing
The program below can be found and fully run when pulling everything from our github repository, where we did all the data processing 

**Description of processing of raw data into scientific evidence:**


In [ ]:
################################################################################################################################################################
"""Plot styling"""

# Increase font sizes across every figure (labels, ticks, legends) so plots
# stay legible once shrunk down to fit the report's column width.
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 16,
})

################################################################################################################################################################
"""Constants"""

g_actual = 9.812                                        # m/s^2 (accepted/reference value for gravitational acceleration)

# y_1 varies per run and is loaded from Data/data.csv instead of being hard-coded here.
d_12 = 7.5                                              # cm  distance between gate 1 and gate 2, fixed for every trial (original y_1 - y_2 = 101.5 - 94)  
y_3 = 1.3                                               # cm

y_1_uncertainty = 0.1                                   # cm  (measuring tape, mm accuracy)
y_2_uncertainty = 0.1                                   # cm  (measuring tape, mm accuracy)
y_3_uncertainty = 0.1                                   # cm  (measuring tape, mm accuracy)

d_12_uncertainty = y_1_uncertainty + y_2_uncertainty    # cm  (d12 = y1 - y2, so uncertainties add)
d_23_uncertainty = y_2_uncertainty + y_3_uncertainty    # cm  (d23 = y2 - y3, so uncertainties add)
d_13_uncertainty = d_12_uncertainty + d_23_uncertainty  # cm  (d13 = d12 - d23, so uncertainties add)

t_uncertainty = 2e-6                                    # s   (single photoelectric sensor reading, microsecond accuracy)
t23_sensor_uncertainty = 2 * t_uncertainty              # s   (t23 = t3 - t2, so the two sensor uncertainties add)

################################################################################################################################################################
"""Functions"""

def _style_axes(ax, include_x_zero=False, include_y_zero=False):
    """
    Apply consistent, rubric-compliant styling to a figure axis:

    - No in-figure title (the caption belongs in the report text, not the
      plot itself).
    - Tick spacing restricted to legible steps of 1, 2 or 5 (via
      MaxNLocator), instead of matplotlib's arbitrary default spacing.
    - A light grid, since it makes it easier to read values off the axes
      precisely -- helpful in both colour and black-and-white printouts.
    - Optionally forces 0 onto an axis where that is a meaningful physical
      reference point (e.g. a time or distance axis that starts at 0).

    ax : a matplotlib Axes object
    """
    ax.set_title('')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 2.5, 5, 10]))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 2.5, 5, 10]))
    if include_x_zero:
        left, right = ax.get_xlim()
        ax.set_xlim(left=min(0, left), right=right)
    if include_y_zero:
        bottom, top = ax.get_ylim()
        ax.set_ylim(bottom=min(0, bottom), top=top)
    ax.grid(True, linewidth=0.4, alpha=0.5)

def load_data(filepath):
    """
    Read the raw trial data from a CSV file.

    Expected columns: run, y_1, t23 (with a header row to skip).

    Returns a list of tuples: (run_id, y_1, time_comp)
    """
    runs = []
    with open(filepath, newline='') as f:
        reader = csv.reader(f)
        next(reader)  # skip header row
        for row in reader:
            if not row:  # skip blank lines (e.g. trailing newline at end of file)
                continue
            run_id, y_1, time_comp = row
            # Convert all values from strings to floats before storing
            runs.append((float(run_id), float(y_1), float(time_comp)))
    return runs

def group_by_y1(runs):
    """
    Group trial timing data by y_1 (i.e. by which distance setting was used).

    runs : list of (run_id, y_1, time_comp) tuples

    Returns a dict mapping y_1 -> list of time_comp values for that y_1.
    """
    groups = {}
    for run_id, y_1, time_comp in runs:
        # setdefault creates an empty list the first time this y_1 is seen
        groups.setdefault(y_1, []).append(time_comp)
    return groups


def get_g(t_23, d_12, d_13):
    """
    Compute g from the timing between gate 2 and gate 3, and the two
    known distances (d_12, d_13).

    t_23  : time between gate 2 and gate 3 (s)
    d_12  : distance between gate 1 and gate 2 (cm)
    d_13  : distance between gate 1 and gate 3 (cm)

    Returns g in m/s^2.
    """
    # Formula derived from constant acceleration kinematics, gives g in cm/s^2
    g_cm_per_s2 = (math.sqrt(2 * d_13) - math.sqrt(2 * d_12)) ** 2 / t_23 ** 2
    # Convert cm/s^2 -> m/s^2
    return g_cm_per_s2 / 100

def get_g_uncertainty(t_23, t23_uncertainty, d_12, d_13):
    """
    Propagate uncertainty in d_12, d_13, and t_23 into an uncertainty on g,
    using exact partial-derivative (first-order) error propagation:
 
        u_g^2 = (dg/dd13)^2 * u_d13^2 + (dg/dd12)^2 * u_d12^2 + (dg/dt23)^2 * u_t23^2
 
    Writing A = sqrt(2*d13), B = sqrt(2*d12), so g = (A-B)^2 / t23^2:
 
        dg/dd13 =  2*(A-B) / (A * t23^2)
        dg/dd12 = -2*(A-B) / (B * t23^2)
        dg/dt23 = -2*g / t23
 
    This treats the three input uncertainties as independent (adding their
    contributions in quadrature), unlike a simple relative-error sum, which
    implicitly assumes worst-case correlated errors and overestimates u_g.
 
    t_23             : time between gate 2 and gate 3 (s)
    t23_uncertainty  : uncertainty in t_23 (s)
    d_12             : distance between gate 1 and gate 2 (cm)
    d_13             : distance between gate 1 and gate 3 (cm)
 
    Returns the uncertainty in g, in m/s^2.
    """
    A = math.sqrt(2 * d_13)
    B = math.sqrt(2 * d_12)
    g_cm_per_s2 = (A - B) ** 2 / t_23 ** 2
 
    # Partial derivatives of g (in cm/s^2) with respect to each input
    dg_dd13 = 2 * (A - B) / (A * t_23 ** 2)
    dg_dd12 = -2 * (A - B) / (B * t_23 ** 2)
    dg_dt23 = -2 * g_cm_per_s2 / t_23
 
    # Combine contributions in quadrature (independent-error propagation)
    g_cm_per_s2_uncertainty = math.sqrt(
        (dg_dd13 * d_13_uncertainty) ** 2 +
        (dg_dd12 * d_12_uncertainty) ** 2 +
        (dg_dt23 * t23_uncertainty) ** 2
    )
 
    # Convert cm/s^2 -> m/s^2
    return g_cm_per_s2_uncertainty / 100

def compute_group_results(groups):
    """
    For each y_1 group: average the trial times, compute g and its
    uncertainty, and print a summary row.

    The uncertainty used for t_23 in this function is the STATISTICAL
    spread of the repeated timing measurements at this y_1 (standard error
    of the mean), combined in quadrature with the fixed sensor uncertainty
    (t23_sensor_uncertainty) -- so even a single trial (n=1) still carries
    the sensor's instrumental uncertainty rather than 0.

    groups : dict mapping y_1 -> list of time_comp values

    Returns two lists (same order, sorted by y_1):
      group_g     : g value per group (m/s^2)
      group_g_unc : uncertainty on g per group (m/s^2)
    """
    group_g = []
    group_g_unc = []

    # Print table header
    print(f"{'y_1 (cm)':>10} {'d_13 (cm)':>10} {'N':>4} {'g (m/s^2)':>14} {'% error':>8}")

    for y_1 in sorted(groups):
        times = groups[y_1]
        n = len(times)
        d_13 = abs(y_1 - y_3)  # distance between gate 1 and gate 3 for this y_1

        # Mean time across all trials at this y_1
        mean_t23 = sum(times) / n

        # Standard error of the mean time (only meaningful with >1 trial)
        if n > 1:
            variance = sum((t - mean_t23) ** 2 for t in times) / (n - 1)
            t23_uncertainty_stat = math.sqrt(variance) / math.sqrt(n)
        else:
            t23_uncertainty_stat = 0.0  # no repeated trials to estimate spread from

        # Total t23 uncertainty: statistical spread and sensor (instrumental)
        # uncertainty combined in quadrature, since they're independent
        t23_uncertainty = math.sqrt(t23_uncertainty_stat ** 2 + t23_sensor_uncertainty ** 2)

        # Compute g and its uncertainty for this group
        g_val = get_g(mean_t23, d_12, d_13)
        g_unc = get_g_uncertainty(mean_t23, t23_uncertainty, d_12, d_13)
        pct_err = abs((g_val - g_actual) / g_actual * 100)

        # Print one row of the summary table
        print(f"{y_1:10.2f} {d_13:10.2f} {n:4d} {g_val:8.4f} +/- {g_unc:.4f} {pct_err:8.2f}")

        group_g.append(g_val)
        group_g_unc.append(g_unc)

    return group_g, group_g_unc


def agreement_metrics(a, u_a, b, u_b):
    """
    Compute the agreement metrics between two measured values, per the
    agreement criterion:

        |v| = |a - b| > 2*sqrt(u_a^2 + u_b^2) = 2*u_v  =>  NOT in good agreement

    a, u_a  : first value and its uncertainty
    b, u_b  : second value and its uncertainty

    Returns (v, u_v, in_agreement), where in_agreement is True if the two
    values ARE in good agreement (v <= u_v).
    """
    v = abs(a - b)
    u_v = 2 * np.sqrt(u_a ** 2 + u_b ** 2)
    return v, u_v, v <= u_v

def check_agreement(a, u_a, b, u_b, label_a='a', label_b='b'):
    """
    Check whether two measured values are in good agreement (see
    agreement_metrics for the criterion used), and print a one-line
    summary of the result.

    a, u_a            : first value and its uncertainty
    b, u_b            : second value and its uncertainty
    label_a, label_b  : optional names used in the printed message

    Prints the result and returns True if a and b ARE in good agreement,
    False otherwise.
    """
    v, u_v, in_agreement = agreement_metrics(a, u_a, b, u_b)

    verdict = "ARE in good agreement" if in_agreement else "are NOT in good agreement"
    comparison = "<=" if in_agreement else ">"
    print(f"{label_a} = {a:.4f} +/- {u_a:.4f}  and  {label_b} = {b:.4f} +/- {u_b:.4f}  "
          f"{verdict}  (|v| = {v:.4f} {comparison} 2u_v = {u_v:.4f})")

    return in_agreement

def print_agreement_table(groups, group_g, group_g_unc, g_fit, g_fit_uncertainty):
    """
    Print two agreement tables:

    1. Every computed g value against the accepted value g_actual: each
       per-height group from the algebraic method (get_g/get_g_uncertainty),
       plus the pooled curve_fit value (g_fit), each against g_actual with
       u_(g_actual) = 0.
    2. Every per-height group g value against the pooled curve_fit value
       (g_fit, with its own uncertainty g_fit_uncertainty) -- checking
       whether the grouped/algebraic results are themselves consistent
       with the pooled fit, independent of the accepted value.

    Uses the same agreement criterion as check_agreement:
        |v| = |a - b| > 2*sqrt(u_a^2 + u_b^2) = 2*u_v  =>  NOT in good agreement

    groups             : dict mapping y_1 -> list of time_comp values
                         (used to get the sorted y_1 values matching
                         group_g/group_g_unc order)
    group_g            : g value per group (m/s^2), sorted by y_1
    group_g_unc        : uncertainty on g per group (m/s^2), sorted by y_1
    g_fit              : pooled curve_fit g value (m/s^2)
    g_fit_uncertainty  : uncertainty on g_fit (m/s^2)
    """
    y1_values = sorted(groups)

    print("Agreement with accepted value g_actual:")
    print(f"{'Source':>18} {'g (m/s^2)':>10} {'u_g':>8} {'v=|g-g_act|':>12} "
          f"{'2u_v':>8} {'Agreement':>12}")

    for y_1, g_val, g_unc in zip(y1_values, group_g, group_g_unc):
        v, u_v, agree = agreement_metrics(g_val, g_unc, g_actual, 0.0)
        label = f"y_1 = {y_1:.2f} cm"
        verdict = "Agree" if agree else "Disagree"
        print(f"{label:>18} {g_val:10.4f} {g_unc:8.4f} {v:12.4f} {u_v:8.4f} {verdict:>12}")

    v, u_v, agree = agreement_metrics(g_fit, g_fit_uncertainty, g_actual, 0.0)
    verdict = "Agree" if agree else "Disagree"
    print(f"{'g_fit (pooled)':>18} {g_fit:10.4f} {g_fit_uncertainty:8.4f} {v:12.4f} "
          f"{u_v:8.4f} {verdict:>12}")

    print()
    print(f"Agreement of each group with the pooled fit "
          f"(g_fit = {g_fit:.4f} +/- {g_fit_uncertainty:.4f} m/s^2):")
    print(f"{'Source':>18} {'g (m/s^2)':>10} {'u_g':>8} {'v=|g-g_fit|':>12} "
          f"{'2u_v':>8} {'Agreement':>12}")

    for y_1, g_val, g_unc in zip(y1_values, group_g, group_g_unc):
        v, u_v, agree = agreement_metrics(g_val, g_unc, g_fit, g_fit_uncertainty)
        label = f"y_1 = {y_1:.2f} cm"
        verdict = "Agree" if agree else "Disagree"
        print(f"{label:>18} {g_val:10.4f} {g_unc:8.4f} {v:12.4f} {u_v:8.4f} {verdict:>12}")


def d23_model(t_23, g_cm_per_s2):
    """
    Model function for curve_fit: predicts d_23 (cm) as a function of t_23,
    for the fixed d_12 spacing, using free-fall kinematics starting from
    rest at point 1 (v1 = 0).

    Coordinate convention: y increases UPWARD (matching how y_1, y_3 were
    actually measured -- y_1 is large/high, y_3 is small/low). Since the
    ball falls downward in this frame, both its velocity and the
    gravitational acceleration are NEGATIVE:

        a_y = -g_cm_per_s2

    v_y2 (signed velocity at point 2) comes from v_y2^2 = 2*g*d_12 (the
    square removes the sign either way), and is negative since the ball is
    moving down:

        v_y2 = -sqrt(2 * g_cm_per_s2 * d_12)

    Position update from point 2 to point 3:

        y3 = y2 + v_y2*t_23 + 0.5*a_y*t_23**2
        y2 - y3 = -(v_y2*t_23 + 0.5*a_y*t_23**2)

    d_23 = y2 - y3 is the positive distance fallen, so:

        d_23 = sqrt(2*g_cm_per_s2*d_12)*t_23 + 0.5*g_cm_per_s2*t_23**2

    t_23        : time between gate 2 and gate 3 (s)
    g_cm_per_s2 : gravitational acceleration magnitude (cm/s^2), the free
                  parameter curve_fit solves for

    Returns predicted d_23 (= y2 - y3, a positive distance) in cm.
    """
    a_y = -g_cm_per_s2                              # acceleration is downward; y increases upward
    v_y2 = -np.sqrt(2 * g_cm_per_s2 * d_12)         # velocity at point 2 (cm/s); negative, moving down
    delta_y = v_y2 * t_23 + 0.5 * a_y * t_23 ** 2   # = y3 - y2 (negative, since the ball fell)
    return -delta_y                                 # d_23 = y2 - y3 = -(y3 - y2), a positive distance

def fit_g_curve_fit(runs):
    """
    Estimate g by pooling every individual trial into a single least-squares
    fit (curve_fit, as introduced in Notebook 5), instead of solving the
    2-point algebraic formula per trial/group and propagating uncertainty
    by hand.

    d_23 = f(t_23; g) is fit directly against ALL trials at once, using the
    signed (y-up, v and g negative) derivation in d23_model(). This makes
    better use of the full dataset than the group-by-group algebraic
    method, and avoids the error amplification that comes from subtracting
    two similar-sized numbers (sqrt(2*d13) - sqrt(2*d12)) in get_g().

    runs : list of (run_id, y_1, time_comp) tuples

    Returns (g_fit, g_fit_uncertainty) in m/s^2.
    """
    all_t23 = np.array([time_comp for run_id, y_1, time_comp in runs])
    all_d23 = np.array([abs(y_1 - y_3) - d_12 for run_id, y_1, time_comp in runs])

    # Least-squares fit of g_cm_per_s2 via curve_fit. p0 is a rough initial
    # guess (in cm/s^2) -- the model is nonlinear in g (it appears under a
    # square root in v_2), so a sensible starting point matters.
    values, covariance = curve_fit(d23_model, all_t23, all_d23, p0=(981.0,))

    # Diagonal of the covariance matrix gives the squared standard error
    # of the fit parameter (see Notebook 5, "Uncertainty in the parameters")
    g_fit_cm_per_s2 = values[0]
    g_fit_uncertainty_cm_per_s2 = np.sqrt(covariance[0, 0])

    # Convert cm/s^2 -> m/s^2
    g_fit = g_fit_cm_per_s2 / 100
    g_fit_uncertainty = g_fit_uncertainty_cm_per_s2 / 100
    pct_err = abs((g_fit - g_actual) / g_actual * 100)

    print(f"g (curve_fit, all {len(runs)} trials pooled) = "
          f"{g_fit:.4f} +/- {g_fit_uncertainty:.4f} m/s^2  ({pct_err:.2f}% error)")

    # Plot the pooled data with the fitted curve on top. A test array is
    # used for a smooth fit line, same approach as Notebook 5.
    t_test = np.linspace(0, 1.1 * max(all_t23), 1000)
    d23_fit = d23_model(t_test, g_fit_cm_per_s2)

    # The fit itself is done in d_23 space (see d23_model), but the plot
    # shows d_13 = d_23 + d_12 instead, since d_13 is the quantity actually
    # referenced elsewhere in the report (Equation~\ref{eq:g} and the
    # results tables). This is just a constant vertical shift by d_12.
    all_d13 = all_d23 + d_12
    d13_fit = d23_fit + d_12

    fig, ax = plt.subplots()
    ax.plot(all_t23, all_d13, marker='o', linestyle='none',
            markerfacecolor='black', markeredgecolor='black', ms=4,
            label='measurements')
    ax.plot(t_test, d13_fit, color='black', linestyle='--', lw=1.5,
            label=f'fit ($g$ = {g_fit:.3f} $\\pm$ {g_fit_uncertainty:.3f} m/s$^2$)')
    ax.set_xlabel('$t_{23}$ (s)')
    ax.set_ylabel('$d_{13}$ (cm)')
    _style_axes(ax, include_x_zero=True, include_y_zero=True)
    ax.legend()
    fig.tight_layout()
    plt.show()

    # Residuals, to sanity-check the fit the same way as Notebook 5
    residuals = all_d23 - d23_model(all_t23, g_fit_cm_per_s2)

    fig, ax = plt.subplots()
    ax.plot(all_t23, residuals, marker='o', linestyle='none',
            markerfacecolor='black', markeredgecolor='black', ms=4,
            label='residual')
    ax.axhline(0, color='black', linestyle='--', lw=1.2, label='zero residual')
    ax.set_xlabel('$t_{23}$ (s)')
    ax.set_ylabel('residual $d_{23}$ (cm)')
    _style_axes(ax, include_x_zero=True)
    ax.legend()
    fig.tight_layout()
    plt.show()

    return g_fit, g_fit_uncertainty


def plot_g_vs_t23(runs):
    """
    Plot g computed individually for every single trial (not grouped/averaged),
    against t23, with error bars, alongside a reference line for the actual g value.

    runs : list of (run_id, y_1, time_comp) tuples
    """
    all_t23 = []
    all_g = []
    all_g_unc = []

    # Compute g and its uncertainty for every individual trial
    for run_id, y_1, time_comp in runs:
        d_13 = abs(y_1 - y_3)
        g_val = get_g(time_comp, d_12, d_13)
        # Single trial, no averaging -- only the sensor's own timing
        # uncertainty applies (no statistical spread to combine with)
        g_unc = get_g_uncertainty(time_comp, t23_sensor_uncertainty, d_12, d_13)
        all_t23.append(time_comp)
        all_g.append(g_val)
        all_g_unc.append(g_unc)

    # Scatter plot of g vs t23, with vertical error bars (the error flags
    # required by the rubric)
    fig, ax = plt.subplots()
    ax.errorbar(all_t23, all_g, yerr=all_g_unc, fmt='o', color='black',
                ecolor='black', markersize=4, capsize=3, label='measured $g$')
    # Horizontal reference line at the accepted value of g -- dashed so it
    # remains distinguishable from the data markers in black and white
    ax.axhline(g_actual, color='black', linestyle='--', lw=1.2,
               label='accepted $g$')

    ax.set_xlabel('$t_{23}$ (s)')
    ax.set_ylabel('$g$ (m/s$^2$)')
    _style_axes(ax, include_x_zero=True)
    ax.legend()
    fig.tight_layout()
    plt.show()

def plot_g_vs_y1(groups, group_g, group_g_unc):
    """
    Plot the averaged g value per y_1 group against y_1 (drop height), to
    check whether apparent g varies with drop height -- in reality this is
    more likely a sign of drag effects than a real change in g.

    groups      : dict mapping y_1 -> list of time_comp values (used to get
                  the sorted y_1 values matching group_g/group_g_unc order)
    group_g     : g value per group (m/s^2), sorted by y_1
    group_g_unc : uncertainty on g per group (m/s^2), sorted by y_1
    """
    y1_values = sorted(groups)

    # Scatter plot of g vs y_1, with vertical error bars (error flags)
    fig, ax = plt.subplots()
    ax.errorbar(y1_values, group_g, yerr=group_g_unc, fmt='o', color='black',
                ecolor='black', markersize=4, capsize=3, label='measured $g$')
    # Horizontal reference line at the accepted value of g
    ax.axhline(g_actual, color='black', linestyle='--', lw=1.2,
               label='accepted $g$')

    ax.set_xlabel('$y_1$ (cm)')
    ax.set_ylabel('$g$ (m/s$^2$)')
    _style_axes(ax)
    ax.legend()
    fig.tight_layout()
    plt.show()

def plot_g_trials_in_group(groups, y_1=None):
    """
    Plot every individual trial's g value (not the group average), each
    with its own propagated uncertainty. The x-axis is trial number rather
    than y_1, since y_1 is fixed within any single group.

    Each point uses only the sensor's timing uncertainty (t23_sensor_uncertainty),
    since these are single-trial values rather than group averages -- no
    statistical spread to combine with, unlike compute_group_results().

    groups : dict mapping y_1 -> list of time_comp values
    y_1    : optional. If given, only that group's trials are plotted (must
             be a key present in `groups`), numbered 1..n within that group.
             If omitted (default), every trial from every group is plotted
             on one figure (e.g. all 15 trials across 3 groups of 5), with
             a continuous trial index across groups and a different marker
             shape per group so they stay distinguishable in black and white.
    """
    # Marker shapes cycle if there are more groups than shapes listed here
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X']

    fig, ax = plt.subplots()

    if y_1 is not None:
        if y_1 not in groups:
            raise KeyError(f"y_1 = {y_1} not found in groups. Available: {sorted(groups)}")
        target_groups = [y_1]
    else:
        target_groups = sorted(groups)

    trial_index = 0  # continuous index across all plotted groups
    all_trial_numbers = []
    for group_i, group_y1 in enumerate(target_groups):
        times = groups[group_y1]
        d_13 = abs(group_y1 - y_3)

        trial_numbers = []
        g_vals = []
        g_uncs = []
        for t_23 in times:
            trial_index += 1
            trial_numbers.append(trial_index)
            g_vals.append(get_g(t_23, d_12, d_13))
            g_uncs.append(get_g_uncertainty(t_23, t23_sensor_uncertainty, d_12, d_13))
        all_trial_numbers.extend(trial_numbers)

        label = 'measured $g$' if y_1 is not None else f'$y_1$ = {group_y1:.2f} cm'
        ax.errorbar(trial_numbers, g_vals, yerr=g_uncs,
                    fmt=markers[group_i % len(markers)], color='black',
                    ecolor='black', markersize=4, capsize=3, label=label)

    # Horizontal reference line at the accepted value of g
    ax.axhline(g_actual, color='black', linestyle='--', lw=1.2,
               label='accepted $g$')

    ax.set_xlabel('Trial number')
    ax.set_ylabel('$g$ (m/s$^2$)')
    ax.set_xticks(all_trial_numbers)
    _style_axes(ax)
    ax.legend()
    fig.tight_layout()
    plt.show()

def plot_g_surface(runs=None, d13_range=None, t23_range=None):
    """
    3D surface plot of g as a function of d_13 and t_23, with d_12 held
    fixed -- these are the only two quantities g actually depends on
    (see get_g()), so this shows the whole "response surface" g is drawn
    from rather than a single 1D slice of it.

    If `runs` is given, the actual per-trial (d_13, t23, g) points are
    scattered on top of the surface so you can see where your real data
    sits relative to the full surface.

    runs       : optional list of (run_id, y_1, time_comp) tuples
    d13_range  : optional (min, max) in cm for the d_13 axis; auto-derived
                 from runs if not given (falls back to a default range
                 above d_12 if runs is also not given)
    t23_range  : optional (min, max) in s for the t_23 axis; auto-derived
                 from runs if not given (falls back to a default range)
    """
    import numpy as np

    if runs is not None:
        d13_vals = [abs(y_1 - y_3) for run_id, y_1, time_comp in runs]
        t23_vals = [time_comp for run_id, y_1, time_comp in runs]
    else:
        d13_vals = []
        t23_vals = []

    # d_13 must exceed d_12 (point 3 is further from point 1 than point 2 is)
    if d13_range is None:
        d13_range = (min(d13_vals) * 0.9, max(d13_vals) * 1.1) if d13_vals \
            else (d_12 * 1.05, d_12 * 3)
    if t23_range is None:
        t23_range = (min(t23_vals) * 0.9, max(t23_vals) * 1.1) if t23_vals \
            else (0.01, 0.5)

    d13_grid = np.linspace(*d13_range, 100)
    t23_grid = np.linspace(*t23_range, 100)
    D13, T23 = np.meshgrid(d13_grid, t23_grid)

    # Vectorized version of get_g()'s formula, with d_12 fixed
    G = (np.sqrt(2 * D13) - np.sqrt(2 * d_12)) ** 2 / T23 ** 2 / 100  # m/s^2

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(D13, T23, G, cmap='viridis', alpha=0.75, edgecolor='none')
    fig.colorbar(surf, shrink=0.6, label='g (m/s^2)')

    if runs is not None:
        g_vals = [get_g(t23, d_12, d13) for t23, d13 in zip(t23_vals, d13_vals)]
        ax.scatter(d13_vals, t23_vals, g_vals, color='red', s=25, label='measured trials')
        ax.legend()

    ax.set_xlabel('$d_{13}$ (cm)')
    ax.set_ylabel('$t_{23}$ (s)')
    ax.set_zlabel('$g$ (m/s$^2$)')
    ax.set_title('')

    plt.tight_layout()
    plt.show()

################################################################################################################################################################
"""Main"""

def main():
    """
    Run the full analysis pipeline:
    load data -> group by distance -> compute g per group (algebraic method)
    -> compute g via a pooled least-squares fit -> plot results.
    """
    
    runs = load_data('Data/data.csv')
    groups = group_by_y1(runs)
    group_g, group_g_unc = compute_group_results(groups)
    g_fit, g_fit_unc = fit_g_curve_fit(runs)
    # print_agreement_table(groups, group_g, group_g_unc, g_fit, g_fit_unc)
    # plot_g_vs_t23(runs)
    plot_g_vs_y1(groups, group_g, group_g_unc)
    plot_g_trials_in_group(groups)  # all 15 trials across every group; pass a specific y_1 to see just one group
    # plot_g_surface(runs)
    # plot_residual_distribution(runs, g_fit)

if __name__ == '__main__':
    main()

 **Describing the pattern in the processed data:**

The gravitational acceleration remains largely constant across different trials from the same height.

The uncertainty of the gravitational acceleration decreases as the height from which the free-fall is tested increases. This is because the numerator of Equation 8 compares two distance measurements of similar magnitude, which results in catastrophic cancellation; a larger dropping distance $d_{13}$ can be used to decrease this effect.

The $g_{fit}$ value is the more accurate estimate, at 9.859 $\pm$ 0.014 $m/s^2$ (0.48\% error), though it is not in statistical agreement with $g_{actual}$, unlike the algebraic results, which are less accurate but do agree due to their larger uncertainties.





# Discussion 

The least-squares fit across all data points provided a much smaller uncertainty ($g_{\text{fit}} = 9.859 \pm 0.014\text{ m/s}^2$). Consequently, the algebraic results agree with $g_{\text{actual}}$ solely because their broad error bars mask deviations, whereas the tight uncertainty in $g_{\text{fit}}$ leads to formal statistical disagreement. Neither method achieved the target $0.1\%$ accuracy or four significant figures, primarily due to manual distance measurement limitations and unmodeled drag effects, which would require an enclosed vacuum setup and higher-grade sensors to resolve.



# Conclusion

The Experiment failed in its goal of finding the value g up to 4 significant digits. The group trial results are still in agreement with the real value, but very inaccurate, while the $g_{fit}$ value is much more accurate, but not in agreement with the true value. This is very probably a consequence of length measurement errors, as well as a possible consequence of the different experiment obstacles we were met with (described in the Notes subsection of the Preparation Section).
